# Can teacher training copy a behaviour the model lacked, faithfully and with Δ fixed?

*E42, `experiments/teacher_training/newskill.py`, 2026-10-28. Record:
`reports/analyses/2026-10-28_analysis_e42-teacher-new-behaviour.md`. Plan:
`reports/plans/2026-10-28_plan_e42-teacher-new-behaviour.md`. Follows E10, E12 and E34.*

*Independent check: `reports/reviews/2026-10-28_review_e42-check.md`, corrections applied.*

## The answer

Yes by the criterion in force, with one caveat that matters. We copied fine-tunes that teach the model a word cipher.
At the cipher's trigger the copies kept the new behaviour on all three tasks. On ordinary text they passed the
harness's reconstruction checks, and Δ stayed untouched. But with Γ frozen, the model built only from the subcomponents
Γ marks important at the trigger loses almost all of it on both random ciphers. No map read so far avoids that.
Unexpectedly, a copy trained on plain Pile text alone reached 0.91 on the one-pair task, where we filed below 0.5.

## What we were trying to test

Teacher training copies a fine-tune into the decomposition. The subcomponents' read and write vectors are trained so
that the model built from them imitates the fine-tuned model, with Γ and Δ frozen. In E34 it kept every fine-tune's
behaviour and passed the adversarial check. But every fine-tune it copied taught something the model already half did.
`GOALS.md` asks what happens when post-training teaches something new. It expects that with Δ fixed, a new behaviour
will run through subcomponents Γ keeps off.

The new behaviour is E10's word cipher. Each window of text holds a marker token that never occurs in the source text,
then a common word, the key. The fine-tuned model must put the key's value next. Each task has 403 training windows
and 97 held-out ones:

- **32 random pairs**: each of 32 keys maps to an arbitrary other word, a substitution cipher the model lacks.
- **1 random pair**: one key and its arbitrary value.
- **Identity control**: each of the 32 keys is followed by itself. The model half does this already, and in E10 its gain
  came from Γ switching on subcomponents that already copy. It checks whether what we see on the random pairs belongs to
  the format or to the behaviour being new.

The fine-tune is E10's, run longer: all 24 matrices, Adam, 1,000 steps. We copied it twice, 1,000 steps each. The copy
on task text trains on the task's training windows. The copy on Pile text trains on 8,192 rows of ordinary Pile text,
which hold no marker. It asks whether the fine-tune's predictions on ordinary text carry the new behaviour.

Each state is read three ways.

- **Behaviour**: the held-out probability of the target at the trigger. The unchanged model reads about 0.00002 on the
  32 random pairs and 0.0002 on the other two.
- **Faithfulness**: the harness's reconstruction checks on 256 Pile validation rows. The verdict rests on two of them.
  A 20-step adversary from one random start searches the masks for the largest output damage. We report what it finds
  at the state over what it finds at the unchanged model, so 1 means no change. The stochastic-mask reading is judged
  the same way. On this L40 card the unchanged model varies by 0.21 across three adversary seeds. So a state passes at
  or below 1.24 and fails at or above 1.81. VPD's sparsity loss was not read.
- **Where the gain sits at the trigger**: the masked-off share. The gain is the rise in the target's log-odds at the
  trigger. The masked-off share is the part of it present when every subcomponent is switched on, but lost when Γ's
  causal-importance values mask them. At 0 the CI-masked model keeps the whole gain; at 1 it keeps none.

Two references sit beside every copy. **Do-nothing** leaves the whole fine-tune in Δ. The **least-squares split**
writes the fine-tune as the least-norm change of the subcomponents and puts the remainder in Δ. It should fail the
check, which shows the check can fail on these tasks.

## What we expected, and why

- **The copy on task text keeps the behaviour and passes, on every task.** Component training on the cipher's labels
  passed on the 32-pair cipher once it had settled. Every task-text copy in E34 passed.
- **Its gain runs through subcomponents Γ keeps off, on both random ciphers.** Label training put 0.75 and 0.86 of the
  gain there in E10. On the identity control we expected a small share, since Γ switches on subcomponents that already
  copy.
- **The copy on Pile text learns nothing of the cipher**, staying below 0.5 on every task. Pile text holds no marker,
  and the fine-tune's training also fitted its predictions at the windows' other positions. We held this with medium
  confidence: E34's Pile-text copies once learnt a behaviour we filed they would not.

## Does the copy keep the behaviour and pass the check, with Δ untouched?

![copy keeps the behaviour and passes](../experiments/teacher_training/figures/e42_copy.png)

**Reading it.** Yes, on all three tasks. In the left panel every copy keeps the target's probability at 0.96 or more,
far above the unchanged model's dashed line. It passes the four reconstruction checks the harness runs on Pile text;
the sparsity loss was not read. In the right panel the copies read 0.96 to 0.99. That is below all three of the
unchanged model's own adversary seeds, which read 1.00 to 1.21 on this scale. The least-squares split fails on every
task, so the check can fail here, though only just on two tasks. Do-nothing reads between the lines on all three. Δ
took none of the copy's change, as it must when frozen.

The copy is the fine-tune's behaviour at the trigger, not the whole fine-tune. Away from the trigger it does 0.35 to
0.40 of the fine-tune's damage. Part of why it passes on Pile text is that it moved the model less there than the
fine-tune did.

## Does the CI-masked model carry the new behaviour?

![CI-masked model on Pile text and at the trigger](../experiments/teacher_training/figures/e42_trigger.png)

**Reading it.** On ordinary text yes; at the trigger no, on both random ciphers. The left panel is the CI-masked check
on Pile text, where every state reads within 4% of the unchanged model. The right panel reads the same masked model at
the trigger, as a mean over 97 held-out windows from one batch order. The copy on 32 random pairs puts 0.96 on the
target with Δ included. With every subcomponent on and Δ left out it puts 0.95 there. Masked by Γ's causal-importance
values, it puts 0.007. On one pair the masked model keeps 0.001 of a full 1.00. On the identity control it keeps 0.93
of 0.98.

With Γ frozen, no map read so far keeps the cipher in the CI-masked model at the trigger. The reasons differ.
Do-nothing leaves the update in Δ, and the least-squares split leaves much of it there. Teacher training, like E10's
label-trained component training, carries it through subcomponents Γ keeps off. So the gap is not specific to teacher
training. Whether any frozen-Γ map can close it is untested: teacher training with the masked losses on the task's
windows has not been tried. The Pile checks are unlikely to see the gap. The marker did not occur in the 1,024 Pile
training rows we scanned, though the harness's 256 validation rows were not scanned.

## Does the copy's gain run through subcomponents Γ keeps off?

![masked-off share at the trigger](../experiments/teacher_training/figures/e42_routes.png)

**Reading it.** Yes on both random ciphers. The masked-off share of the copy's gain at the trigger is 0.84 on the large
cipher and 0.92 on the one-pair cipher. Label-trained component training read 0.75 and 0.86 in E10, at earlier steps,
so the copy goes the same way. On the identity control, the filed positive control, the share is 0.31. That is just
above the 0.25 line we filed as "not masked-off", but still far from the ciphers. There most of the gain comes from Γ
switching on subcomponents that already copy, as with label training. Do-nothing reads below zero by construction: its
whole update sits in Δ, outside the masked model. Caveat: one batch order, and how far these shares move between batch
orders is unmeasured.

## Can ordinary Pile text alone teach the copy the cipher?

![copy trained on Pile text, and the fine-tune's predictions on Pile text](../experiments/teacher_training/figures/e42_pile.png)

**Reading it.** On the one-pair task the copy trained on Pile text reached 0.91 by step 1,000, where we filed below
0.5, and passed the check at 1.10. On the other two tasks it stayed far below 0.5, as filed: 0.02 and 0.001. What it
can have learnt from text without the marker is the fine-tune's key-to-value bigram, which also answers the held-out
windows. The right panel shows a candidate cause. In plain Pile text, with no marker, the fine-tune puts 0.31 on the
value right after the one-pair key, where the unchanged model puts 0.0001. That key occurs 182 times in the first
1,024 rows. For 32 pairs the fine-tune's leak is far smaller, at 0.005. Whether the leak is what teaches the copy is
untested. Caveat: 0.91 sits only 0.01 above the line. Pile-text copies moved by up to 0.03 between batch orders in
E34, so a rerun could land "between", which would still contradict the filed prediction.

In [1]:
# The scored numbers behind the figures, from the three results files (no model).
import json
from pathlib import Path

res = Path("../experiments/teacher_training/results")
print(f"{'task':8s} {'copy P':>7s} {'check':>6s} {'masked-off':>10s} {'Pile-only P':>11s}")
for t in ("rand32", "rand1", "ident32"):
    r = json.loads((res / f"teacher_training_e42_{t}.json").read_text())
    a, b = r["A"]["1000"], r["B"]["1000"]
    print(f"{t:8s} {a['p_target']:7.3f} {a['pgd']:6.2f} {a['share_e_gate']:10.2f} {b['p_target']:11.3f}")

task      copy P  check masked-off Pile-only P
rand32     0.964   0.96       0.84       0.019
rand1      1.000   0.99       0.92       0.911
ident32    0.983   0.96       0.31       0.001


## What we learned

Teacher training copies the behaviour of a fine-tune that teaches something the model lacked. On three cipher tasks it
keeps the behaviour at the trigger, passes the harness's reconstruction checks on ordinary text and leaves Δ alone.
With Γ frozen, the copy carries the new behaviour through subcomponents Γ keeps off at the trigger. That is what
`GOALS.md`'s "create" branch expects when Δ may not move, and label-trained component training showed it too. The
criterion in force, reading A of 2026-10-19, says the paper's checks decide, so E42 passes. `GOALS.md`'s mechanism
section, which the file marks as not deciding, says the weak version holds when the masked-off share is small; by that
E42 would fail. The files settle which text decides. The open question is whether this case, the first to separate
them clearly, should change reading A, which the user left open to change.

## How compelling is the evidence

- **One draw of everything.** One batch order, one fine-tune seed, and per state one adversary start of 20 steps on 256
  Pile rows. In E34 a second random start once read a fine-tune six times higher. The adversarial ratios are lower
  bounds, so a pass may be lenient.
- **The stochastic ratio separates nothing here.** It reads 0.87 to 0.94 for every state, do-nothing and the
  least-squares split included, and carries none of the verdicts.
- **The L40's band is wide**, from 1.24 to 1.81; an H100's is about a third as wide. No copy depends on it: every copy
  reads 1.10 or less.
- **The check's power is thin on two tasks.** The least-squares split fails at 1.83 and 1.86, just over the line.
- **The trigger readings** are means over 97 held-out windows at one position, logged to three decimals. How far the
  masked-off share moves between batch orders is unmeasured.
- **The fine-tune reproduced E10's** to four decimals on 32 pairs and the identity control. On one pair that check could
  not fail: E10 read 0.998, and any working fine-tune lands within 0.02 of it.
- **Scope**: the 67M model, three tasks of one format, one fine-tune recipe.

## Surprises

- **Something unexpected happened that I cannot fully explain.** The copy trained on Pile text alone reached 0.91 on
  the one-pair task and passed. We filed it would stay below 0.5. A candidate cause is visible but untested: the
  fine-tune shows the pair in ordinary text.
- **The fine-tune leaks its map into plain text more than the plan expected.** The plan expected at most 10 times the
  unchanged model's probability. It read 146 times on the large cipher, and about 3,000 times on the one-pair cipher.
  Reported, not scored.
- **The identity control's masked-off share landed at 0.31**, just above the 0.25 line, so "mixed" where we leaned "not
  masked-off". A narrow miss: at step 100 it read 0.25.

## What next

1. **List the subcomponents that carry the copy's gain** on the random ciphers, from the saved states on CPU, at no
   cost. Suppose a handful carry it and Γ is near zero on them at the trigger. Then a small change to Γ is the natural
   repair; if hundreds share it, it is not.
2. **Masked teacher training on the cipher windows**, the existing masked-loss variant from E37, with Γ frozen. If the
   CI-masked model then keeps the cipher at the trigger, the gap belongs to the maps tried, not to a frozen Γ.
3. **Test the Pile-only one-pair result** by dropping every Pile row that holds the key. If the copy still learns, the
   fine-tune's leak at the key is not the cause.
4. **The user's decision** on whether this case should change reading A.

## Scorecard

Filed in `experiments/teacher_training/results/predictions_e42.json` before any run, unedited. Scored by the run, under
`score` in each results file, and checked here against the filed rules.

| id | claim | rule | read | verdict |
|---|---|---|---|---|
| C1 | unchanged model under the Pile harness | W₀ at its known values, floor ≤ stop | floor 0.209, pass | pass |
| G1 | fine-tune at step 10 reproduces E10 | within 0.02 of E10 | 0.5847 / 0.9595 / 0.9982 | pass ×3 (one pair cannot fail) |
| P1 rand32 | task-text copy faithful | `state_verdict` at 1000 | P 0.964, ratios 0.960 / 0.936 | **pass**, as predicted |
| P1 rand1 | task-text copy faithful | same | P 1.000, ratios 0.989 / 0.941 | **pass**, as predicted |
| P1 ident32 | task-text copy faithful | same | P 0.983, ratios 0.957 / 0.929 | **pass**, as predicted |
| P2 rand32 | masked-off | e_gate ≥ 0.5 | 0.839 | **masked-off**, lean held |
| P2 rand1 | masked-off | e_gate ≥ 0.5 | 0.918 | **masked-off**, lean held |
| P2 ident32 | not masked-off | e_gate ≤ 0.25 | 0.309 | **mixed**: lean missed narrowly |
| P3 rand32 | Pile-text copy: neither | P < 0.5 | P 0.019 | **neither**, lean held |
| P3 rand1 | Pile-text copy: neither | P < 0.5 | P 0.911, ratios 1.095 / 0.883, pass | **learnt: reversed** |
| P3 ident32 | Pile-text copy: neither | P < 0.5 | P 0.0013 | **neither**, lean held |
| power | least-squares split fails | verdict fail | 1.977 / 1.826 / 1.855 | power on all three |

## Not run

- A second batch order or fine-tune seed; a second adversary start or a longer adversary on any state.
- VPD's sparsity loss on any state.
- An H100 rerun: the decision tree sends inconclusive copies there, and no copy was inconclusive.
- The list of subcomponents that carry the copy's gain, a branch of the plan, from the saved states.
- Masked teacher training on the task's windows, or any frozen-Γ map limited to the subcomponents Γ switches on.
- The CI-masked KL at the trigger itself, rather than the target's probability; the unchanged model's own trigger cells
  were not kept in the results.
- The marker count in the harness's 256 Pile validation rows.
- The Pile-only one-pair copy without the key's rows; Pile-text copies longer than 1,000 steps.
- A Pile reading of the copies; an SGD fine-tune or copy; a LoRA fine-tune; E10's step-10 fine-tune; the larger model.

## Provenance

| task | cluster job | run, 2026-10-28 | results file |
|---|---|---|---|
| 32 random pairs | 40166 | 12:02-12:23 | `results/teacher_training_e42_rand32.json` |
| identity control | 40194 | 12:12-12:35 | `results/teacher_training_e42_ident32.json` |
| 1 random pair | 40195 | 12:14-12:36 | `results/teacher_training_e42_rand1.json` |

MATS cluster, NVIDIA L40, debug QoS, no cost. Logs in `cluster/logs/commute-teacher-e42-<job>.out`, saved states in
`experiments/teacher_training/results/large/e42/<task>/`, figures from `experiments/teacher_training/plot_e42.py`.

## Amendment 2: does either masked term, or both at the trigger only, hold still with nothing to copy?

*2026-10-28. Record: `reports/analyses/2026-10-28_analysis_e42-amend2.md`. Plan:
`reports/plans/2026-10-28_plan_e42-teacher-new-behaviour.md` → "Amendment 2". Follows Amendment 1,
`reports/analyses/2026-10-28_analysis_e42-amend1.md`.*

**The answer.** We tried three simpler versions of masked teacher training on the 32-key cipher, each run first with
nothing to copy. Every version broke the worst-case check on ordinary text while the random-mask check stayed level, so
the plan stopped before any copy was made. So no training of this kind has yet kept a new behaviour in the parts the
decomposition switches on. The pending choice on whether the test should change stands as it was.

### What we were trying to test

The copies above keep the cipher, but the CI-masked model at the trigger does not. The CI-masked model is the model
rebuilt from the subcomponents alone, each scaled by its causal importance $g$ from Γ, with Δ left out. Amendment 1
asked whether teacher training could be made to put the cipher there while Γ stays frozen. It added two terms to the
loss. One matches the CI-masked student to the teacher. The other matches the stochastic-masked student, whose masks
are $g + (1-g)r$ with $r$ uniform on [0, 1], drawn afresh at every position. Before the cipher, it ran the recipe with
the teacher set to the unchanged model, so there was nothing to copy. That control broke the adversarial check, so the
cipher copy never ran.

This amendment reruns that control three ways, to find which term does the damage. Each runs 1,000 steps on the 32-key
cipher's training windows, with Amendment 1's batches and random masks:

- the CI-masked term alone;
- the stochastic term alone;
- both terms read at the trigger position only, run only if both single-term controls fail.

A usable recipe should leave the decomposition where it was. Each control is read by the Pile harness, as above: the
adversarial ratio passes at 1.24 or below and fails at 1.81 or above on this card. If a control held, the cipher copy
would run with it. If all three failed, the plan stops.

### What we expected, and why

We expected the CI-masked term alone to fail, with medium confidence. It trains only the case where every mask is Γ's
value. The switched-off subcomponents can then change to cancel the switched-on ones' changes, and an adversary that
sets masks in between undoes the cancelling. We leaned towards the stochastic term alone passing, at low confidence,
since random masks on every subcomponent should penalise that cancelling. Against that, E37's control trained with
VPD's own stochastic loss and still failed. We expected the trigger-only recipe to fail, at low confidence: it asks
less, but it still has no worst-case term.

Step 1 of every control reproduced Amendment 1's step 1 to every printed digit, so the batches and masks are the same.

### With nothing to copy, did any masked recipe keep the Pile checks?

![every masked control fails the adversarial check; the stochastic check stays level](../experiments/teacher_training/figures/e42_amend2_controls.png)

**Reading it.** No: all three new controls failed the adversarial check, and so did Amendment 1's. In the left panel
the unchanged model's three adversary seeds sit at 1.0 to 1.21. E42's copy, trained without masked terms on the same
windows, sits at 0.96. Every masked recipe sits between 11.7 and 48.7, six to twenty-seven times the fail line. In the
right panel every state's random-mask ratio stays between 0.94 and 1.04, far under its line. So each recipe left the
decomposition as good as before on average and broke it in the worst case. The stochastic term alone did the least
damage and the CI-masked term alone the most. That ordering is one batch order and one adversary start, and a
control's spread across batch orders is unmeasured.

### What did each control train, and what did the target model give up?

![masked terms fall about a third; the unmasked term drifts alike under every recipe](../experiments/teacher_training/figures/e42_amend2_curves.png)

**Reading it.** Each recipe did what it was built to do. In the left panel each trained term fell to about 0.7 of its
starting level on the full windows, as in Amendment 1. The trigger-only terms fell to about 0.55. The term left
untrained fell too, by less. In the right panel the unmasked term measures how far the target model has moved from the
teacher, which is the unchanged model. It starts at 0.0016 nats, jumps to about 0.09 within 50 steps and ends near
0.04. All four recipes trace the same curve, while their adversarial ratios range from 12 to 49. So this drift does not
tell the recipes' damage apart.

### What we learned

No masked teacher training recipe tried on the cipher windows with Γ frozen holds still with nothing to copy. Four
recipes failed, each at least six times past the line. So no frozen-Γ map has been shown able to keep a new behaviour
in the CI-masked model. None has been shown unable to either, since every attempt broke its own control first. For
`GOALS.md`'s fork between a new capability and a brittle decomposition, E42's finding stands. The cipher sits in
subcomponents Γ keeps off at the trigger, and whether that is forced is untested. The plan's tree stops the masked line
here, and the pending choice on the test is left as it is.

### How compelling is the evidence

The failures are far outside any noise. The smallest is six times the fail line, and the unchanged model's adversary
seeds spread by 0.21. Both 128-row blocks failed in every control. The comparison between recipes rests on one task,
one batch order, one adversary start and one weight. The stochastic check is blind by construction to changes that
cancel on average, which is why it read level throughout. Why the recipes break the worst case is not tested. Candidate
causes are the missing worst-case term that VPD trains with, and fitting the masked students on a narrow set of windows
with no sparsity loss to push back.

### Surprises

The stochastic term alone failed, reversing our low-confidence lean towards a pass. It read 11.7, six times the fail
line: the mildest recipe, but nowhere near holding. E37's failure with VPD's stochastic loss, filed as evidence against
the lean, was the better guide. Smaller: the CI-masked term alone did nearly twice the damage of both terms together,
which may sit inside a control's unmeasured spread.

### What next

- No run follows: the tree stops here. The pending choice on the test stays as it is, with its option of first
  training such a copy marked as tried and failed. If the user asks for the stricter test, the next step is letting Γ
  change.
- Reopening the masked line would mean adding an adversarial term matched to the teacher. E37 suggests that may not be
  enough, and it needs new code and a review.
- At no cost, the three saved control states could be re-read with two more adversary seeds. That would show whether
  the ordering among recipes is real; it would not change the stop.

### Amendment 2 scorecard

| id | claim in words | rule | read | verdict |
|---|---|---|---|---|
| C1 | the unchanged model under the Pile harness is its own floor | `w0_check` | floor 0.20878, as in job 40338 | pass |
| G2 | the trainer's unmasked forward is the model's | ≤ 0.01 nats | 0.00163 | pass |
| C_ci | the CI-masked term alone holds still | either Pile verdict "fail" or NaN → fail | adversarial 48.74, stochastic 1.016 | **fail**, as leaned |
| C_st | the stochastic term alone holds still | as C_ci | adversarial 11.71, stochastic 1.011 | **fail**, lean "pass" reversed |
| C_trig | both terms at the trigger only hold still | as C_ci | adversarial 34.04, stochastic 1.038 | **fail**, as leaned: the tree stops |
| P1, P2 | the copy keeps the cipher in the CI-masked model; it passes | Amendment 1's rules | no copy ran | not scored |

### Amendment 2: not run

- The cipher copy with any masked recipe; the teacher regeneration and its check against E10; the reference re-reads.
- rand1 and ident32; a second batch order; other weights; a second adversary start on any control.
- An adversarial term in the loop; the masked terms on Pile rows; updating Γ; VPD's sparsity loss; the larger model.

MATS cluster, NVIDIA L40, debug QoS, job 40497, 14:32-14:56, no cost. Log
`cluster/logs/commute-teacher-e42a2-40497.out`; results
`experiments/teacher_training/results/teacher_training_e42_amend2_rand32.json`; saved states in
`experiments/teacher_training/results/large/e42_amend2/rand32/`; figures from
`experiments/teacher_training/plot_e42_amend2.py`.